## Trích embedding cho HuBERT và WavLM

In [6]:
import torch
import torchaudio
import os
import gc
import glob
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import Wav2Vec2FeatureExtractor, WavLMModel, HubertModel, Wav2Vec2Model

# ==============================================================================
# 1. DATASET TỐI ƯU
# ==============================================================================
class TestSpeakerDataset(Dataset):
    def __init__(self, folder_path):
        self.folder_path = folder_path
        # Chỉ lấy file .wav
        self.file_paths = glob.glob(os.path.join(folder_path, "**", "*.wav"), recursive=True)
        # Sắp xếp theo độ dài (ước tính qua dung lượng) giúp giảm padding thừa trong batch
        self.file_paths.sort(key=lambda x: os.path.getsize(x))
        
    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        try:
            waveform, sr = torchaudio.load(path)
            if sr != 16000:
                waveform = torchaudio.functional.resample(waveform, sr, 16000)
            
            # Chuyển mono
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0)
            else:
                waveform = waveform.squeeze(0)
                
            rel_path = os.path.relpath(path, self.folder_path)
            return waveform, rel_path
        except Exception as e:
            print(f"Lỗi file {path}: {e}")
            return torch.zeros(16000), "error"

def collate_fn_test(batch):
    # Lọc bỏ các file lỗi
    batch = [b for b in batch if b[1] != "error"]
    waveforms, rel_paths = zip(*batch)
    return list(waveforms), list(rel_paths)

# ==============================================================================
# 2. CẤU HÌNH CHO RTX 4060 (8GB)
# ==============================================================================
INPUT_FOLDER = r"D:\Study\7-SP26\DATxSLP\Test set O\test-O"
OUTPUT_DIR = r"D:\Study\7-SP26\DATxSLP\Test set O\Data after embedding"
MODEL_KEY = "hubert" 
BATCH_SIZE = 16  # Có thể tăng lên 24-32 nếu file âm thanh ngắn (<10s)

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_map = {
    "wavlm": (WavLMModel, "microsoft/wavlm-base"),
    "hubert": (HubertModel, "facebook/hubert-base-ls960"),
    "wav2vec2": (Wav2Vec2Model, "facebook/wav2vec2-base-960h")
}

model_class, repo = model_map[MODEL_KEY]
processor = Wav2Vec2FeatureExtractor.from_pretrained(repo)

print(f"Loading {MODEL_KEY.upper()}...")
# Sử dụng torch_dtype=torch.float16 để tiết kiệm VRAM và tăng tốc trên 4060
model = model_class.from_pretrained(
    repo, 
    output_hidden_states=True,
    torch_dtype=torch.float16, 
    attn_implementation="eager" # Sử dụng Scaled Dot Product Attention (nhanh & nhẹ hơn eager)
).to(device).eval()

dataset = TestSpeakerDataset(INPUT_FOLDER)
# Với 32GB RAM DDR5, đặt num_workers=4 để load data nhanh hơn
dataloader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    collate_fn=collate_fn_test, 
    shuffle=False, 
    num_workers=0, 
    pin_memory=True
)

all_embeddings_dict = {}

# ==============================================================================
# 3. TRÍCH XUẤT (SỬ DỤNG FP16)
# ==============================================================================
print(f"\n--- TRÍCH XUẤT TRÊN {device.upper()} (FP16 MODE) ---")

with torch.inference_mode():
    for waveforms, rel_paths in tqdm(dataloader, desc="Processing"):
        
        # SỬA TẠI ĐÂY: Chuyển list các tensor thành list các numpy array 1D
        # Điều này giúp processor nhận diện đúng cấu trúc để thực hiện padding
        waveforms_input = [w.numpy() for w in waveforms]

        try:
            # Thực hiện padding và chuyển sang tensor
            inputs = processor(
                waveforms_input, 
                sampling_rate=16000, 
                return_tensors="pt", 
                padding=True
            ).to(device)
            
            # Chuyển sang FP16 để tối ưu VRAM cho 4060
            inputs = {k: v.to(torch.float16) if torch.is_floating_point(v) else v for k, v in inputs.items()}

            outputs = model(**inputs)
            
            # Trích xuất hidden states
            # [Layers, Batch, Time, Dim]
            stacked = torch.stack(outputs.hidden_states) 
            
            # Mean pooling theo chiều Time (dim=2) -> [Layers, Batch, Dim]
            # Sau đó permute về [Batch, Layers, Dim]
            pooled = stacked.mean(dim=2).permute(1, 0, 2).cpu()
            
            for j in range(len(rel_paths)):
                all_embeddings_dict[rel_paths[j]] = pooled[j]

        except torch.cuda.OutOfMemoryError:
            # Nếu vẫn OOM (do batch có file quá dài), giải phóng cache và chạy đơn lẻ
            torch.cuda.empty_cache()
            for w, p_path in zip(waveforms_input, rel_paths):
                # Xử lý đơn lẻ 1 file
                inp = processor(w, sampling_rate=16000, return_tensors="pt").to(device)
                inp = {k: v.to(torch.float16) if torch.is_floating_point(v) else v for k, v in inp.items()}
                
                out = model(**inp)
                p = torch.stack(out.hidden_states).mean(dim=2).permute(1, 0, 2).cpu()
                all_embeddings_dict[p_path] = p[0]
            torch.cuda.empty_cache()

# Lưu kết quả
save_file = os.path.join(OUTPUT_DIR, f"Embeddings_{MODEL_KEY}.pt")
torch.save(all_embeddings_dict, save_file)

# Clear bộ nhớ
del model, processor
gc.collect()
torch.cuda.empty_cache()

print(f"✅ Xong! Đã lưu {len(all_embeddings_dict)} file vào: {save_file}")

Loading HUBERT...

--- TRÍCH XUẤT TRÊN CUDA (FP16 MODE) ---


Processing: 100%|██████████| 1112/1112 [05:00<00:00,  3.69it/s]


✅ Xong! Đã lưu 17786 file vào: D:\Study\7-SP26\DATxSLP\Test set O\Data after embedding\Embeddings_hubert.pt


## Trích embedding cho Wav2Vec2

In [ ]:
import torch
import torchaudio
import os
import gc
import glob
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import WavLMModel, HubertModel, Wav2Vec2Model

# ==============================================================================
# 1. DATASET TỐI ƯU (Tích hợp chuẩn hóa để chống lỗi NaN)
# ==============================================================================
class TestSpeakerDataset(Dataset):
    def __init__(self, folder_path):
        self.folder_path = folder_path
        # Chỉ lấy file .wav
        self.file_paths = glob.glob(os.path.join(folder_path, "**", "*.wav"), recursive=True)
        # Sắp xếp theo độ dài giúp giảm padding thừa trong batch
        self.file_paths.sort(key=lambda x: os.path.getsize(x))
        
    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        try:
            waveform, sr = torchaudio.load(path)
            if sr != 16000:
                waveform = torchaudio.functional.resample(waveform, sr, 16000)
            
            # Chuyển mono
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0)
            else:
                waveform = waveform.squeeze(0)
                
            # Chuẩn hóa Zero-mean, Unit-variance (Tương đương việc HuggingFace Processor làm)
            # Rất quan trọng để tránh NaN khi qua các lớp LayerNorm của Wav2Vec2
            mean = waveform.mean()
            var = waveform.var(unbiased=False)
            waveform = (waveform - mean) / torch.sqrt(var + 1e-7)
                
            rel_path = os.path.relpath(path, self.folder_path)
            return waveform, rel_path
        except Exception as e:
            print(f"Lỗi file {path}: {e}")
            return torch.zeros(16000), "error"

def collate_fn_test(batch):
    # Lọc bỏ các file lỗi
    batch = [b for b in batch if b[1] != "error"]
    waveforms, rel_paths = zip(*batch)
    
    # Pad sequence siêu tốc bằng PyTorch (nhanh hơn HuggingFace Processor)
    padded_waveforms = torch.nn.utils.rnn.pad_sequence(waveforms, batch_first=True, padding_value=0.0)
    
    # Tạo Attention Mask để model bỏ qua các vùng padding
    lengths = torch.tensor([len(w) for w in waveforms])
    max_len = padded_waveforms.shape[1]
    attention_mask = torch.arange(max_len).expand(len(lengths), max_len) < lengths.unsqueeze(1)
    
    return padded_waveforms, attention_mask.long(), list(rel_paths)

# ==============================================================================
# 2. CẤU HÌNH CHO RTX 4060 (8GB)
# ==============================================================================
INPUT_FOLDER = r"D:\Study\7-SP26\DATxSLP\Test set O\test-O"
OUTPUT_DIR = r"D:\Study\7-SP26\DATxSLP\Test set O\Data after embedding"
MODEL_KEY = "wav2vec2"  # Chạy thử cho wav2vec2
BATCH_SIZE = 16

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_map = {
    "wav2vec2": (Wav2Vec2Model, "facebook/wav2vec2-base-960h")
}

model_class, repo = model_map[MODEL_KEY]

print(f"Loading {MODEL_KEY.upper()}...")
# SỬ DỤNG torch.bfloat16 ĐỂ TRÁNH LỖI NAN MÀ VẪN TIẾT KIỆM VRAM
model = model_class.from_pretrained(
    repo, 
    output_hidden_states=True,
    torch_dtype=torch.bfloat16, 
    attn_implementation="sdpa" # Flash Attention (nhanh & nhẹ hơn eager)
).to(device).eval()

dataset = TestSpeakerDataset(INPUT_FOLDER)
dataloader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    collate_fn=collate_fn_test, 
    shuffle=False, 
    num_workers=0, # Đặt num_workers 4 hoặc 8 để load file nhanh hơn
    pin_memory=True
)

all_embeddings_dict = {}

# ==============================================================================
# 3. TRÍCH XUẤT (SỬ DỤNG BFLOAT16)
# ==============================================================================
print(f"\n--- TRÍCH XUẤT TRÊN {device.upper()} (BFLOAT16 MODE) ---")

with torch.inference_mode():
    for waveforms, attention_mask, rel_paths in tqdm(dataloader, desc="Processing"):
        
        # Đưa tensor lên GPU với định dạng bfloat16
        waveforms = waveforms.to(device, dtype=torch.bfloat16, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)

        try:
            outputs = model(input_values=waveforms, attention_mask=attention_mask, output_hidden_states=True)
            
            # Trích xuất hidden states [Layers, Batch, Time, Dim]
            stacked = torch.stack(outputs.hidden_states) 
            
            # Mean pooling theo chiều Time (dim=2) -> [Layers, Batch, Dim]
            # Sau đó permute về [Batch, Layers, Dim] và cast về .float() (float32) trước khi lưu
            pooled = stacked.mean(dim=2).permute(1, 0, 2).cpu().float()
            
            for j in range(len(rel_paths)):
                all_embeddings_dict[rel_paths[j]] = pooled[j]

        except torch.cuda.OutOfMemoryError:
            # Xử lý đơn lẻ nếu có file bất thường làm quá tải RAM
            torch.cuda.empty_cache()
            for i in range(len(waveforms)):
                w = waveforms[i].unsqueeze(0)
                mask = attention_mask[i].unsqueeze(0)
                
                out = model(input_values=w, attention_mask=mask, output_hidden_states=True)
                p = torch.stack(out.hidden_states).mean(dim=2).permute(1, 0, 2).cpu().float()
                all_embeddings_dict[rel_paths[i]] = p[0]
            torch.cuda.empty_cache()

# Lưu kết quả
save_file = os.path.join(OUTPUT_DIR, f"Embeddings_{MODEL_KEY}.pt")
torch.save(all_embeddings_dict, save_file)

# Clear bộ nhớ
del model
gc.collect()
torch.cuda.empty_cache()

print(f"✅ Xong! Đã lưu {len(all_embeddings_dict)} file vào: {save_file}")

## Kiểm tra NaN

In [7]:
import torch
import glob

# Đường dẫn tới file pt
shard_file = r"D:\Study\7-SP26\DATxSLP\Test set O\Data after embedding\Embeddings_hubert.pt"

try:
    # Load data
    data = torch.load(shard_file, map_location='cpu')
    
    total_nan = 0
    total_inf = 0
    total_files = len(data)
    files_with_issues = []

    # Duyệt qua từng file trong dictionary
    for file_path, embedding in data.items():
        nan_in_file = torch.isnan(embedding).sum().item()
        inf_in_file = torch.isinf(embedding).sum().item()
        
        total_nan += nan_in_file
        total_inf += inf_in_file
        
        if nan_in_file > 0 or inf_in_file > 0:
            files_with_issues.append(file_path)

    print(f"--- KẾT QUẢ KIỂM TRA ---")
    print(f"File: {shard_file}")
    print(f"Tổng số file âm thanh đã trích xuất: {total_files}")
    print(f"Tổng số giá trị NaN: {total_nan}")
    print(f"Tổng số giá trị Inf: {total_inf}")
    
    if len(files_with_issues) > 0:
        print(f"⚠️ Có {len(files_with_issues)} file bị lỗi (NaN/Inf).")
        print(f"Danh sách file lỗi (5 file đầu): {files_with_issues[:5]}")
    else:
        print("✅ Tuyệt vời! Không có giá trị NaN hoặc Inf nào.")

except Exception as e:
    print(f"❌ Lỗi khi đọc hoặc xử lý file: {e}")

--- KẾT QUẢ KIỂM TRA ---
File: D:\Study\7-SP26\DATxSLP\Test set O\Data after embedding\Embeddings_hubert.pt
Tổng số file âm thanh đã trích xuất: 17786
Tổng số giá trị NaN: 0
Tổng số giá trị Inf: 0
✅ Tuyệt vời! Không có giá trị NaN hoặc Inf nào.
